# YOLOv8n Inside Bar Pattern Trainer â€” Colab Glue

This notebook is **only** environment setup + invocation glue for `scripts/pattern_scanner/train.py`.
All training logic lives in `train.py` so it runs identically on Colab or locally â€” see `08-CONTINUE-HERE.md`.

## Prereqs (do these locally before opening Colab)

1. Generate the dataset locally: `python -m scripts.pattern_scanner.generate_training_data --seed 42`
2. Zip it: `tar -czf training_dataset.tgz _dev/training_dataset/`
3. Upload `training_dataset.tgz` + `yolov8n.pt` to a known Google Drive folder.
   Suggested path: `MyDrive/jun-xian-portfolio/phase-08/`
4. Push your latest `main` to GitHub (notebook clones from `origin/main`).

## Runtime

**Runtime â†’ Change runtime type â†’ T4 GPU.** If "no GPU available", retry in a few hours.
Free tier idle-disconnects at ~90 min, so write `runs/` to Drive (Cell 4) for resumability.

## Cell 1 â€” environment + GPU check

In [ ]:
# Confirm we got a T4 (or better). If this prints "command not found", you're on CPU runtime â€” bail.
!nvidia-smi

In [ ]:
# Install everything Cell 5 (train.py) and Cell 7 (renderer + yfinance for holdout fixture) need.
# Numpy is pinned because Colab's preinstalled numpy can clash with ultralytics build.
# matplotlib + Pillow are preinstalled on Colab; yfinance + mplfinance are not.
%pip install -q 'ultralytics>=8.4.0,<9.0' 'numpy<2.0' 'yfinance>=0.2.65' 'mplfinance>=0.12.10b0'

In [ ]:
import torch
import ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available(), 'device_count', torch.cuda.device_count())

## Cell 2 â€” mount Drive (resumability + artifact return path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CHANGE this to your actual folder if different.
DRIVE_FOLDER = '/content/drive/MyDrive/jun-xian-portfolio/phase-08'
!ls -la {DRIVE_FOLDER}

## Cell 3 â€” clone repo, fetch dataset + base weights

In [ ]:
%cd /content
# CHANGE if your repo URL is different.
!git clone --depth=1 https://github.com/jxgoh004/jun-xian-project.git
%cd jun-xian-project

# Pull dataset + base weights from Drive into the working tree.
!cp {DRIVE_FOLDER}/training_dataset.tgz .
!cp {DRIVE_FOLDER}/yolov8n.pt .
!tar -xzf training_dataset.tgz
!ls -la _dev/training_dataset/ | head -20
!cat _dev/training_dataset/data.yaml

In [ ]:
# Quick sanity check â€” manifest counts so you can confirm the dataset is what you expected.
import json
manifest = json.load(open('_dev/training_dataset/dataset_manifest.json'))
print(json.dumps({k: manifest[k] for k in manifest if k != 'samples'}, indent=2))
print('total samples in manifest:', len(manifest.get('samples', [])))

## Cell 4 â€” W4 human-verify gate (manual pause)

**Before running Cell 5, confirm:**

- `nvidia-smi` showed a T4 (or better) GPU
- Manifest `positives` count matches what you expected from the local dry-run
- `data.yaml` paths point at `_dev/training_dataset/images/{train,val}` (relative paths inside the repo)
- Hyperparameters in `scripts/pattern_scanner/train.py` are what you want: `imgsz=640`, `fliplr=0.0`, `seed=42`, default 100 epochs with `patience=20`

If anything is off, edit `train.py` here in Colab (or earlier in your local repo, push, re-clone) **before** running Cell 5.

### Optional: redirect ultralytics `runs/` to Drive for disconnect resilience

If you're worried about idle disconnect, run the cell below to symlink `runs/` to Drive. Then if the runtime dies you can resume training from `last.pt` instead of starting over.

In [ ]:
import os
os.makedirs(f'{DRIVE_FOLDER}/runs', exist_ok=True)
if not os.path.lexists('runs'):
    os.symlink(f'{DRIVE_FOLDER}/runs', 'runs')
!ls -la runs

## Cell 5 â€” train + export ONNX (the long one, ~20-40 min on T4)

Calls the project's own `train.py`. All hyperparameters are locked there.
Watch stdout for per-epoch progress, `[train] best mAP50: X.XXXX`, and `[export] OPSET: 12`.

In [ ]:
!python -m scripts.pattern_scanner.train --seed 42 --epochs 100 --patience 20

## Cell 6 â€” W5 human-verify gate (manual pause)

Read the printed `best mAP50` and `best mAP50-95`. Open `models/training_summary.json`:

In [ ]:
import json
summary = json.load(open('models/training_summary.json'))
print(json.dumps(summary, indent=2))

# Sanity checks the unit tests will enforce locally:
assert summary['opset'] == 12
assert summary['imgsz'] == 640
assert summary['fliplr'] == 0.0

import os
size = os.path.getsize('models/inside_bar_v1.onnx')
print(f'ONNX size: {size:,} bytes ({size/1e6:.1f} MB)')
assert 1_000_000 <= size <= 50_000_000, f'unexpected ONNX size: {size}'

**Decision:** If `best_map50` looks reasonable for your dataset (e.g., > 0.5 is typical for a single-class detector with a few thousand positives â€” adjust expectation per your actual sample count), proceed to Cell 7.

If it looks bad (e.g., < 0.2): consider re-running Cell 5 with `--epochs 200` for more training time, or revisit the dataset (positives count, label quality, style randomization range).

If you want to give up: Cell 7 is still safe to run â€” it ships the artifacts you have. You can always retrain later.

## Cell 7 â€” render holdout fixture (post-cutoff positive)

Plan 05 (verify_onnx.py) consumes a held-out positive PNG that is NOT in the training set â€”
it must come from a detection with `confirmation_date >= TRAIN_TEST_CUTOFF` (= '2024-01-01').

This cell finds one and renders it via the project's renderer.

In [ ]:
import json
from pathlib import Path
import yfinance as yf
from scripts.pattern_scanner.detector import detect
from scripts.pattern_scanner.renderer import STYLES, render, compute_bbox_normalized
from scripts.pattern_scanner.split_config import TRAIN_TEST_CUTOFF

# Try a handful of mega-cap tickers â€” at least one should yield a post-cutoff positive.
for ticker in ['MSFT', 'AAPL', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM']:
    df = yf.Ticker(ticker).history(period='10y', auto_adjust=True)
    if df.empty:
        continue
    df.index = df.index.tz_localize(None)
    detections = [d for d in detect(df, ticker) if d.confirmation_date >= TRAIN_TEST_CUTOFF]
    if detections:
        d = detections[0]
        # Take the 60-bar window ending at confirmation
        end = d.confirmation_bar_index + 1
        start = max(0, end - 60)
        window = df.iloc[start:end]
        png_bytes = render(window, STYLES[0])
        bbox = compute_bbox_normalized(window, d.mother_bar_index - start, d.confirmation_bar_index - start)
        Path('tests/fixtures').mkdir(parents=True, exist_ok=True)
        Path('tests/fixtures/known_positive_holdout.png').write_bytes(png_bytes)
        meta = {
            'ticker': ticker,
            'confirmation_date': d.confirmation_date,
            'style': STYLES[0].name,
            'cutoff': TRAIN_TEST_CUTOFF,
            'post_cutoff': True,
            'bbox_cx_cy_w_h': list(bbox),
        }
        Path('tests/fixtures/known_positive_holdout.json').write_text(json.dumps(meta, indent=2))
        print('[fixture] rendered for', ticker, 'on', d.confirmation_date)
        break
else:
    print('[fixture] WARNING â€” no post-cutoff positive found in mega-cap set; widen the search')

## Cell 8 â€” copy artifacts back to Drive (the deliverables)

Five files go back to your laptop:

- `models/inside_bar_v1.onnx`
- `models/training_summary.json`
- `models/dataset_manifest.json` (real-run copy)
- `tests/fixtures/known_positive_holdout.png`
- `tests/fixtures/known_positive_holdout.json`

In [ ]:
import shutil
out = f'{DRIVE_FOLDER}/artifacts'
import os; os.makedirs(out, exist_ok=True)
for src in ['models/inside_bar_v1.onnx',
            'models/training_summary.json',
            'models/dataset_manifest.json',
            'tests/fixtures/known_positive_holdout.png',
            'tests/fixtures/known_positive_holdout.json']:
    if os.path.exists(src):
        shutil.copy(src, out)
        print(f'[copy] {src} -> {out}')
    else:
        print(f'[skip] {src} missing')
!ls -la {out}

## Done

On your laptop:

1. Download the 5 files from `MyDrive/jun-xian-portfolio/phase-08/artifacts/` into the matching local paths.
2. `git add scripts/pattern_scanner/train.py models/inside_bar_v1.onnx models/training_summary.json models/dataset_manifest.json tests/fixtures/known_positive_holdout.png tests/fixtures/known_positive_holdout.json tests/test_onnx_round_trip.py` then commit per task per Plan 08-04.
3. Run `/gsd-execute-phase 8 --wave 4` to land the GSD-side commits + SUMMARY.md.
4. Run `/gsd-execute-phase 8 --wave 5` for the verify_onnx clean-venv round-trip.